SPDX-FileCopyrightText: 2025 Dan J. Bower <dbower@eaps.ethz.ch>

SPDX-License-Identifier: GPL-3.0-or-later

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from bedroc import debug_logger
from bedroc.isotope_anomalies.core import get_group_data
from bedroc.pca import Analyzer

logger = debug_logger()
logger.setLevel(logging.INFO)

savefig_kwargs = {"dpi": 300, "bbox_inches": "tight", "format": "pdf"}
"""Figure options for savefig"""

# Nucleosynthetic isotope anomalies

Citation:

**Sossi, Paolo A., and Bower, Dan J. (2026), Homogeneous accretion of the Earth in the inner Solar System, Nature Astronomy**

In [ ]:
# Set a random seed for reproducibility
RANDOM_SEED = 123

Get available group data

In [ ]:
group_data_dict = get_group_data()
group_data_dict.keys()

Select the group to analyse

In [ ]:
select_group = "all"
# select_group = "heavy"
# select_group = "iron-peak"
# select_group = "lithophile"
# select_group = "siderophile"
# select_group = "iron-set"
# select_group = "chromium-set"
# select_group = "vesta_3_systems"
# select_group = "vesta_4_systems"

Create an output directory for all data files and figures

In [ ]:
output_directory = Path(f"out_isotope_anomalies_{select_group}")
output_directory.mkdir(parents=True, exist_ok=True)

Get the group data

In [ ]:
group_data = group_data_dict[select_group]

# Show the data
group_data.data.df_raw

Plot the Pearson correlation coefficient

In [ ]:
ax = group_data.data.plot_pearson_correlation_coefficient()
outpath = output_directory / Path(f"pearson_correlation_coefficient.{savefig_kwargs['format']}")
ax.figure.savefig(outpath, **savefig_kwargs)  # pyright: ignore

Run the Bayesian PCA

In [ ]:
group_data.run_bayesian_pca(random_seed=RANDOM_SEED)

Export a summary of the inference data to Excel

In [ ]:
group_data.to_excel(output_directory)

Export the inference data to a pickle file

In [ ]:
group_data.to_pickle(output_directory)

Create the analyzer, generate plots, and output summary statistics

In [ ]:
analyzer = Analyzer(group_data.model, group_data.idata)

In [ ]:
_, summary_by_feature = analyzer.plot_explained_variance_by_feature()

In [ ]:
summary_by_feature

In [ ]:
_, summary_by_factor = analyzer.plot_explained_variance_by_factor()

In [ ]:
summary_by_factor

In [ ]:
fig, ax = plt.subplots()

_ = group_data.plot_pca(ax, plot_legend=True, include_title=True)

Plot the reconstructed observations and output summary statistics

In [ ]:
figures, recon_stats = group_data.plot_reconstructed_observations(
    reconstruction_only=False, random_seed=RANDOM_SEED
)

for data, figure in zip(group_data.data.data_names, figures):
    outpath = output_directory / Path(f"reconstructed_{data}.{savefig_kwargs['format']}")
    figure.savefig(outpath, **savefig_kwargs)

In [ ]:
recon_stats.to_excel(output_directory / Path("reconstructed_observations.xlsx"))

Compute the inferred isotope anomaly of Mercury and Venus. The values of the latent factors and their standard deviations are given by the York regression presented in the manuscript.

In [ ]:
data_names = ["Venus", "Mercury"]
latent_factor_means = np.array([[-2.88, -4.58], [-3.24, -6.40]])
latent_factors_stds = np.array([[0.0, 0.02], [0.03, 0.18]])

In [ ]:
figures, pred_stats = group_data.plot_predicted_observations(
    latent_factor_means=latent_factor_means,
    latent_factor_stds=latent_factors_stds,
    data_names=data_names,
)

for data, figure in zip(data_names, figures):
    outpath = output_directory / Path(f"prediction_{data}.{savefig_kwargs['format']}")
    figure.savefig(outpath, **savefig_kwargs)

In [ ]:
pred_stats.to_excel(output_directory / Path("venus_mercury_predictions.xlsx"))